# 共有コーパスアーティファクトの Hub への切り出し(Promoting Canonical Corpus Artifacts)

## 目的

複数のトピックで再利用しうるコーパスアーティファクトを、それぞれ独立した Hugging Face
Hub の Dataset リポジトリへ切り出すための運用スクリプトである(`theories/`の特定の
トピックにも`apps/`の特定のアプリにも属さない、リポジトリ運用スクリプト)。

現時点で扱うコーパスは以下の 2 つだが、`CORPUS_SPECS`(次のセル)にスペックを
追加するだけで別の言語・別のマニフェストのコーパスを追加できる構造にしている
(`promote_canonical_tokenizers.ipynb`の`ARTIFACTS`辞書と同じ考え方)。

| キー | 言語 | マニフェスト | 記事数 | 用途 |
|---|---|---|---|---|
| en | 英語(en) | `en_009_scaling.json` | 1000 | 009(スケーリング則)の学習グリッド用 |
| ja | 日本語(ja) | `ja_006_pretraining.json` | 80 | 006(小型 GPT の事前学習)の日本語条件用 |

BPE の学習・言語モデルの学習を伴わない、純粋な取得・分割・アップロードの処理であり
GPU を要しないため、本番スケールのままローカルで構築・検証する。Hugging Face Hub
への実際のアップロード呼び出しのみ、`HF_TOKEN`の認証を要する操作としてガードし
(`DRY_RUN`・`IN_COLAB`)、Google Colab で行う。

**対象外**: `theories/`のノートブック自体は変更しない。各コーパスを実際に使う側
(例: 009 の`load_english_wikipedia_corpus_with_fallback()`呼び出し)の更新は、
本ノートブックとは別に行う。006 は既に完成済みのノートブックであり、日本語コーパスを
本アーティファクトから読み込むように変更することは対象外(再実行コストが高いため)。
`load_japanese_wikipedia_corpus()`(`src/data/text.py`)は、将来の日本語トピックの
ために用意した関数であり、現時点で本ノートブック以外からの呼び出し元はない。</cell id="f48034bf">

In [1]:
# 環境セットアップ(Google Colab)
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/kojikojiprg/ai-theories.git
    %cd ai-theories
    !pip install uv -q
    !uv pip install --system -r requirements.txt
# ローカル(Jupyter)実行時は、リポジトリルートで起動していればそのまま動く。

Cloning into 'ai-theories'...
remote: Enumerating objects: 584, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 584 (delta 43), reused 51 (delta 25), pack-reused 502 (from 1)
Receiving objects: 100% (584/584), 6.69 MiB | 14.74 MiB/s, done.
Resolving deltas: 100% (299/299), done.
/content/ai-theories
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 76.6 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 52 packages in 659ms
Prepared 28 packages in 57.77s
Uninstalled 10 packages in 1.21s
Installed 28 packages in 496ms
 - click==8.5.0
 + click==8.4.2
 + cuda-bindings==13.3.1
 + cuda-pathfinder==1.6.0
 + cuda-toolkit==13.0.3.0
 - filelock==3.32.4
 + filelock==3.32.2
 - fsspec==2025.3.0
 + fsspec==2026.7.0
 - matplotlib==3.10.0
 + matplotlib==3.11.1
 - numpy==2.1.3
 + numpy==2.5.2
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvidia-cuda-runtime==13.0.96
 +

In [2]:
import json
import subprocess
import time
from pathlib import Path

from src.data.text import (
    load_english_wikipedia_corpus,
    load_japanese_wikipedia_corpus,
    split_train_val_text,
    upload_corpus_artifact_to_hub,
)

DRY_RUN = False  # Claude Code はこの True 側のみ実行する(Colab で DRY_RUN=False に切り替えるとアップロードが実行される)

ROOT = Path(".")
OUTPUT_DIR = ROOT / ".cache" / "promote_canonical_corpora"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 将来、別の言語・別のマニフェストのコーパスを追加する場合は、このリストにスペックを
# 追加するだけでよい。リポジトリ名にはマニフェストの版数・記事数を含めない
# (CLAUDE.md の命名規則。異なるマニフェストが必要になった場合はブランチで管理する、
# 各アーティファクトのデータセットカードに明記する)。
CORPUS_SPECS = [
    {
        "key": "en",
        "language": "en",
        "repo_id": "kojikojiprg/ai-theories-corpus-en",
        "manifest": "en_009_scaling.json",
        "manifest_article_count": 1000,
        "validation_ratio": 0.05,
        # 009 本体(theories/02_pretraining/009_scaling_laws.ipynb)と同じキャッシュ
        # ディレクトリを指定し、記事ごとのキャッシュ(wikipedia_en_articles/)を共有する。
        "cache_dir": ROOT / ".cache" / "009_corpus",
        "loader": load_english_wikipedia_corpus,
        "usage_note": "009(スケーリング則)の学習グリッド用",
    },
    {
        "key": "ja",
        "language": "ja",
        "repo_id": "kojikojiprg/ai-theories-corpus-ja",
        "manifest": "ja_006_pretraining.json",
        "manifest_article_count": 80,
        "validation_ratio": 0.05,
        # 006 本体(theories/02_pretraining/006_pretraining_small_gpt.ipynb)と同じ
        # キャッシュディレクトリを指定し、記事ごとのキャッシュ(wikipedia_ja_articles/)を
        # 共有する。
        "cache_dir": ROOT / ".cache" / "006_corpus",
        "loader": load_japanese_wikipedia_corpus,
        "usage_note": "006(小型 GPT の事前学習)の日本語条件用",
    },
]

## コーパスの取得・分割・JSON の組み立て

各コーパスについて、`loader`(記事数を縮小せず本番のマニフェスト全体で呼び出す)で
コーパスを取得し、`validation_ratio`で訓練・検証分割の決定性を確認したうえで、
アップロード用の`corpus.json`を組み立てる。

In [3]:
corpus_artifacts = {}

for spec in CORPUS_SPECS:
    key = spec["key"]
    print(f"=== {key} ===")

    t0 = time.time()
    raw_text, fetch_metadata = spec["loader"](spec["cache_dir"], return_metadata=True)
    elapsed = time.time() - t0
    print(f"取得時間: {elapsed:.1f} s ({elapsed / 60:.1f} 分)")

    raw_bytes = len(raw_text.encode("utf-8"))
    print(f"raw_text: {len(raw_text):,} 文字 / {raw_bytes:,} バイト")
    print(
        f"取得できた記事数: {fetch_metadata['fetched_article_count']} / "
        f"{fetch_metadata['manifest_article_count']}"
    )
    if fetch_metadata["skipped_articles"]:
        print(f"[警告] {len(fetch_metadata['skipped_articles'])} 記事の取得に失敗した:")
        for a in fetch_metadata["skipped_articles"]:
            print(f"  - {a['title']}: {a['reason']}")
    else:
        print("すべての記事を取得できた(スキップなし)")

    assert fetch_metadata["manifest_article_count"] == spec["manifest_article_count"], (
        f"{key}: マニフェスト記事数が期待({spec['manifest_article_count']})と一致しない: "
        f"{fetch_metadata['manifest_article_count']}"
    )

    train_text, val_text = split_train_val_text(raw_text, spec["validation_ratio"])
    train_text_2, val_text_2 = split_train_val_text(raw_text, spec["validation_ratio"])
    assert train_text == train_text_2 and val_text == val_text_2, f"{key}: 分割が決定的でない"
    print(f"[OK] {key}: split_train_val_text の決定性を確認した")
    print(f"train_text: {len(train_text):,} 文字, val_text: {len(val_text):,} 文字")

    source_commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    ).stdout.strip()

    corpus_payload = {
        "language": spec["language"],
        "manifest": spec["manifest"],
        "manifest_article_count": fetch_metadata["manifest_article_count"],
        "fetched_article_count": fetch_metadata["fetched_article_count"],
        "skipped_articles": fetch_metadata["skipped_articles"],
        "raw_text": raw_text,
        "validation_ratio": spec["validation_ratio"],
        "raw_bytes": raw_bytes,
        "source_commit": source_commit,
    }

    artifact_dir = OUTPUT_DIR / key
    artifact_dir.mkdir(parents=True, exist_ok=True)
    corpus_json_path = artifact_dir / "corpus.json"
    corpus_json_path.write_text(json.dumps(corpus_payload, ensure_ascii=False), encoding="utf-8")
    print(
        f"corpus.json を書き出した: {corpus_json_path} "
        f"({corpus_json_path.stat().st_size:,} バイト)"
    )

    corpus_artifacts[key] = {
        "spec": spec,
        "payload": corpus_payload,
        "artifact_dir": artifact_dir,
        "corpus_json_path": corpus_json_path,
    }
    print()

=== en ===
[警告] en: 目標 1000 記事中 999 記事のみ取得できた(スキップ 1 件)。詳細は .cache/009_corpus/wikipedia_en_fetch_metadata.json を参照。
取得時間: 6313.5 s (105.2 分)
raw_text: 60,247,276 文字 / 60,580,536 バイト
取得できた記事数: 999 / 1000
[警告] 1 記事の取得に失敗した:
  - List of foreign footballers in top leagues of former Yugoslavia: RuntimeError("en revid=1370486518: 応答 JSON に 'parse' キーがない(error={'code': 'nosuchrevid', 'info': 'There is no revision with ID 1370486518.', 'docref': 'See https://en.wikipedia.org/w/api.php for API usage. Subscribe to the mediawiki-api-announce mailing list at &lt;https://lists.wikimedia.org/postorius/lists/mediawiki-api-announce.lists.wikimedia.org/&gt; for notice of API deprecations and breaking changes.'})")
[OK] en: split_train_val_text の決定性を確認した
train_text: 57,234,913 文字, val_text: 3,012,363 文字
corpus.json を書き出した: .cache/promote_canonical_corpora/en/corpus.json (61,398,950 バイト)

=== ja ===
ja: 80/80 記事を取得した
取得時間: 497.0 s (8.3 分)
raw_text: 8,955,329 文字 / 24,575,245 バイト
取得できた記事数: 80 / 80
すべての記事を取

## データセットカードの作成

各アーティファクトについて、由来(マニフェスト名・記事数)・取得できなかった記事・
ライセンス(`cc-by-sa-4.0`、他の成果物との差異の理由)・バージョン管理の方針
(異なるマニフェストが必要になった場合は新しいリポジトリを作らずブランチで管理する)
を含むデータセットカードを作成する。

In [4]:
def build_dataset_card(payload: dict, spec: dict) -> str:
    skipped = payload["skipped_articles"]
    skipped_section = (
        "\n".join(f"- {a['title']}: {a['reason']}" for a in skipped)
        if skipped
        else "なし(すべての記事を取得できた)"
    )
    return f"""---
language: {payload["language"]}
license: cc-by-sa-4.0
tags:
- ai-theories
- corpus
- wikipedia
---

# ai-theories {payload["language"]} コーパス

`ai-theories`(https://github.com/kojikojiprg/ai-theories)プロジェクトの成果物。
{spec["usage_note"]}のコーパス。

## ライセンスについての注記

このデータセット自体のライセンスは **クリエイティブ・コモンズ 表示-継承 4.0 国際
(CC BY-SA 4.0)** である。`ai-theories`の他の成果物(トークナイザなど)は
`cc-by-nc-4.0`を採用しているが、本データセットの内容はフリー百科事典
『ウィキペディア(Wikipedia)』の本文そのもの(wikitext を平文に変換したのみで、
内容は改変していない)であり、Wikipedia 本文自体のライセンス(CC BY-SA 4.0、
表示・継承の条件)を継承する必要があるため、別のライセンスとしている。

## 由来

`src/data/wikipedia_manifests/{payload["manifest"]}`
(記事タイトル・リビジョン ID を固定したマニフェスト、
{payload["manifest_article_count"]} 記事)から、Wikimedia API
(`action=parse`、`oldid`でリビジョンを指定)で取得した。タイトルとリビジョン ID を
両方固定しているため、取得時点によらず同一の入力が得られる。

- マニフェスト記事数: {payload["manifest_article_count"]}
- 取得できた記事数: {payload["fetched_article_count"]}

**取得できなかった記事**:

{skipped_section}

## バージョン管理についての注記

このリポジトリ名(`{spec["repo_id"]}`)には、マニフェストの版数や記事数を含めない。
**将来、同じ言語で異なるマニフェスト(記事数や選定基準が異なるコーパス)が必要に
なった場合は、新しいリポジトリを作らずブランチで管理する。** 現在このリポジトリが
対象としているマニフェストは`{payload["manifest"]}`({payload["manifest_article_count"]}
記事)であり、これは上記「由来」節と本カードの記載からのみ判別できる(リポジトリ名
からは判別できない)。

研究・教育目的で構築したものであり、品質保証は行っていない。商用・実運用での利用は
想定しない。

## 構成

`corpus.json` は以下のフィールドを含む。

- `raw_text`: 取得できた記事本文を連結した全文
- `validation_ratio`: {payload["validation_ratio"]}(`src/data/text.py`の
  `split_train_val_text()`で使う訓練・検証分割の比率)
- `raw_bytes`: `raw_text`の UTF-8 バイト数({payload["raw_bytes"]:,})
- `source_commit`: 取得時点の`ai-theories`リポジトリのコミットハッシュ
  (`{payload["source_commit"]}`)
"""


for key, artifact in corpus_artifacts.items():
    card = build_dataset_card(artifact["payload"], artifact["spec"])
    card_path = artifact["artifact_dir"] / "README.md"
    card_path.write_text(card, encoding="utf-8")
    artifact["dataset_card_path"] = card_path
    print(f"{key}: データセットカードを書き出した: {card_path}")

en: データセットカードを書き出した: .cache/promote_canonical_corpora/en/README.md
ja: データセットカードを書き出した: .cache/promote_canonical_corpora/ja/README.md


## アップロード(Google Colab で `DRY_RUN=False` として実行する)

`DRY_RUN=True` の間はアップロードを一切呼び出さない。`upload_corpus_artifact_to_hub()`
(`src/data/text.py`)は`DRY_RUN=False and IN_COLAB`の場合のみ、
`google.colab.userdata.get("HF_TOKEN")`で取得したトークンを使って呼び出す
(トークナイザ切り出し・008 と同じ認証パターン)。Claude Code はこのセルにおいて、
トークンの入力・環境変数への設定・実際のアップロード実行を一切行わない。

In [5]:
if not DRY_RUN and IN_COLAB:
    from google.colab import userdata

    token = userdata.get("HF_TOKEN")
    for key, artifact in corpus_artifacts.items():
        repo_id = artifact["spec"]["repo_id"]
        print(f"アップロード中: {key} -> {repo_id}")
        upload_corpus_artifact_to_hub(
            repo_id=repo_id,
            corpus_json_path=artifact["corpus_json_path"],
            dataset_card_text=artifact["dataset_card_path"].read_text(encoding="utf-8"),
            token=token,
        )
        print(f"アップロード完了: https://huggingface.co/datasets/{repo_id}")
else:
    print(
        "アップロードをスキップした(DRY_RUN=True またはローカル実行のため)。"
        "本番実行は Google Colab で DRY_RUN=False の状態で行うこと。"
    )

アップロード中: en -> kojikojiprg/ai-theories-corpus-en


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...al_corpora/en/corpus.json:  26%|##5       | 15.9MB / 61.4MB            

アップロード完了: https://huggingface.co/datasets/kojikojiprg/ai-theories-corpus-en
アップロード中: ja -> kojikojiprg/ai-theories-corpus-ja


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...al_corpora/ja/corpus.json:   4%|3         |  957kB / 24.7MB            

アップロード完了: https://huggingface.co/datasets/kojikojiprg/ai-theories-corpus-ja


## まとめ

- コーパスを本番スケール(マニフェスト全体)のままローカルで取得した(記事単位の
  取得失敗はスキップして継続する仕組みが機能することを確認済み)。
- `split_train_val_text()`が決定的であることを確認した。
- アップロード用の`corpus.json`・データセットカードを`.cache/`に書き出した。
- **アップロードは実行していない。** こうじさんが Google Colab で `DRY_RUN=False`
  に切り替えて実行する必要がある。
- 将来コーパスを追加する場合は、`CORPUS_SPECS`にスペックを 1 件追加するだけでよい。